In [ ]:
# ★★このセルを実行後セッションを再起動してください★★再起動後は実行しなくてOK
# numpy_version=2.0.2 tensorflow_version=2.18.0
# 1. numpy と tensorflow を完全アンインストール
!pip uninstall -y keras tensorflow
# 2. keras はインストールせず、tensorflow のみ
!pip install tensorflow==2.18.0 --quiet

Found existing installation: keras 2.15.0
Uninstalling keras-2.15.0:
  Successfully uninstalled keras-2.15.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-text 2.18.1 requires tensorflow<2.19,>=2.18.0, but you have tensorflow 2.15.0 which is incompatible.
tensorflow-decision-forests 1.11.0 requires tensorflow==2.18.0, but you have tensorflow 2.15.0 which is incompatible.
tf-keras 2.18.0 requires tensorflow<2.19,>=2.18, but you have tensorflow 2.15.0 which is incompatible.


In [ ]:
# # tensorflow_version_test
# # 1. numpy と tensorflow を完全アンインストール
# !pip uninstall -y numpy tensorflow keras
# # 2. numpy のバージョンを 1.25.2 に固定（TF2.15との相性良）
# !pip install numpy==1.25.2 --quiet
# # 3. keras はインストールせず、tensorflow のみ
# !pip install tensorflow==2.15.0 --quiet

In [ ]:
# GPUの割り当てを確認（Colab無料枠ならcommand not foundがでるけど気にせず実行しなくて大丈夫）
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Mon Apr 21 10:46:26 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# ドライブのマウント
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
%cd drive/MyDrive/ssd_tf2_lowlevel_study/ssd_keras/
!pwd

[Errno 2] No such file or directory: 'drive/MyDrive/ssd_tf2_lowlevel_study/ssd_keras/'
/content/drive/MyDrive/ssd_tf2_lowlevel_study/ssd_keras
/content/drive/MyDrive/ssd_tf2_lowlevel_study/ssd_keras


In [ ]:
threshold = 0.6 #モデル判定のしきい値
# weights_name = '33_epoch-30_step-0000_loss-1.9178_valloss-2.9057'
weights_name = 'weights_epoch-13_loss-2.90'
model_save_path = '/content/drive/MyDrive/ssd_tf2_lowlevel_study/ssd_keras/checkpoints/'
test_file_path = '/content/drive/MyDrive/ssd_workspace/VOCdevkit/test/VOC2007/JPEGImages/'

In [ ]:
import cv2
from tensorflow import keras
from tensorflow.keras.applications.imagenet_utils import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt
import numpy as np
from imageio import imread
import tensorflow as tf
from glob import glob
from math import ceil
from tqdm import tqdm
# print("tensorflow :", tf.__version__)
# print("matplotlib.pyplot :", matplotlib.__version__)
# print("cv2 :", cv2.__version__)
# print("numpy :", np.__version__)
# print("imageio :", imageio.__version__)
# print("tqdm :", tqdm.__version__)

from ssd_tf2 import SSD300
from ssd_utils_tf2 import BBoxUtility

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 8)
plt.rcParams['image.interpolation'] = 'nearest'

np.set_printoptions(suppress=True)

# config = tf.ConfigProto()
# config.gpu_options.per_process_gpu_memory_fraction = 0.45
# set_session(tf.Session(config=config))

PIL : 11.1.0


AttributeError: module 'os' has no attribute '__version__'

In [ ]:
voc_classes = ['Aeroplane', 'Bicycle', 'Bird', 'Boat', 'Bottle',
               'Bus', 'Car', 'Cat', 'Chair', 'Cow', 'Diningtable',
               'Dog', 'Horse','Motorbike', 'Person', 'Pottedplant',
               'Sheep', 'Sofa', 'Train', 'Tvmonitor']
NUM_CLASSES = len(voc_classes) + 1

In [ ]:
input_shape=(300, 300, 3)
model = SSD300(input_shape, num_classes=NUM_CLASSES)
model.load_weights(f'{model_save_path}{weights_name}.weights.h5')
bbox_util = BBoxUtility(NUM_CLASSES, nms_thresh=0.3, top_k=400)

In [ ]:
inputs = []
images = []

test_data_paths = glob(f"{test_file_path}*.jpg")


# for test_data_path in test_data_paths:
#     img = image.load_img(test_data_path, target_size=(300, 300))
#     img = image.img_to_array(img)
#     images.append(imread(test_data_path))
#     inputs.append(img.copy())
# inputs = preprocess_input(np.array(inputs))

In [ ]:
batch_size = 20
print(f"バッチサイズ：{batch_size} テストデータ数：{len(test_data_paths)}")
num_batches = ceil(len(test_data_paths) / batch_size)

for batch_idx in range(num_batches):
    start = batch_idx * batch_size
    end = start + batch_size
    batch_paths = test_data_paths[start:end]

    inputs = []
    images = []

    for test_data_path in batch_paths:
        img = image.load_img(test_data_path, target_size=(300, 300))
        img_array = image.img_to_array(img)
        inputs.append(img_array.copy())
        images.append(imread(test_data_path))  # 表示用など

    inputs = preprocess_input(np.array(inputs))

    # このバッチだけで推論・処理などを行う
    preds = model.predict(inputs, batch_size=1, verbose=1)
    results = bbox_util.detection_out(preds)

    # 結果の可視化や保存などをここで行う
    for i, img in enumerate(tqdm(images)):
        # Parse the outputs.
        det_label = results[i][:, 0]
        det_conf = results[i][:, 1]
        det_xmin = results[i][:, 2]
        det_ymin = results[i][:, 3]
        det_xmax = results[i][:, 4]
        det_ymax = results[i][:, 5]

        # Get detections with confidence higher than 0.6.
        top_indices = [i for i, conf in enumerate(det_conf) if conf >= threshold]

        top_conf = det_conf[top_indices]
        top_label_indices = det_label[top_indices].tolist()
        top_xmin = det_xmin[top_indices]
        top_ymin = det_ymin[top_indices]
        top_xmax = det_xmax[top_indices]
        top_ymax = det_ymax[top_indices]

        colors = plt.cm.hsv(np.linspace(0, 1, 21)).tolist()

        plt.imshow(img / 255.)
        currentAxis = plt.gca()

        for i in range(top_conf.shape[0]):
            xmin = int(round(top_xmin[i] * img.shape[1]))
            ymin = int(round(top_ymin[i] * img.shape[0]))
            xmax = int(round(top_xmax[i] * img.shape[1]))
            ymax = int(round(top_ymax[i] * img.shape[0]))
            score = top_conf[i]
            label = int(top_label_indices[i])
            label_name = voc_classes[label - 1]
            display_txt = '{:0.2f}, {}'.format(score, label_name)
            coords = (xmin, ymin), xmax-xmin+1, ymax-ymin+1
            color = colors[label]
            currentAxis.add_patch(plt.Rectangle(*coords, fill=False, edgecolor=color, linewidth=2))
            currentAxis.text(xmin, ymin, display_txt, bbox={'facecolor':color, 'alpha':0.5})

        plt.show()

    print(f"✅ Batch {batch_idx+1}/{num_batches} 完了！")

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
'''
inputs = []
images = []
img_path = './pics/fish-bike.jpg'
img = image.load_img(img_path, target_size=(300, 300))
img = image.img_to_array(img)
images.append(imread(img_path))
inputs.append(img.copy())
img_path = './pics/cat.jpg'
img = image.load_img(img_path, target_size=(300, 300))
img = image.img_to_array(img)
images.append(imread(img_path))
inputs.append(img.copy())
img_path = './pics/boys.jpg'
img = image.load_img(img_path, target_size=(300, 300))
img = image.img_to_array(img)
images.append(imread(img_path))
inputs.append(img.copy())
img_path = './pics/car_cat.jpg'
img = image.load_img(img_path, target_size=(300, 300))
img = image.img_to_array(img)
images.append(imread(img_path))
inputs.append(img.copy())
img_path = './pics/car_cat2.jpg'
img = image.load_img(img_path, target_size=(300, 300))
img = image.img_to_array(img)
images.append(imread(img_path))
inputs.append(img.copy())
inputs = preprocess_input(np.array(inputs))
'''

"\ninputs = []\nimages = []\nimg_path = './pics/fish-bike.jpg'\nimg = image.load_img(img_path, target_size=(300, 300))\nimg = image.img_to_array(img)\nimages.append(imread(img_path))\ninputs.append(img.copy())\nimg_path = './pics/cat.jpg'\nimg = image.load_img(img_path, target_size=(300, 300))\nimg = image.img_to_array(img)\nimages.append(imread(img_path))\ninputs.append(img.copy())\nimg_path = './pics/boys.jpg'\nimg = image.load_img(img_path, target_size=(300, 300))\nimg = image.img_to_array(img)\nimages.append(imread(img_path))\ninputs.append(img.copy())\nimg_path = './pics/car_cat.jpg'\nimg = image.load_img(img_path, target_size=(300, 300))\nimg = image.img_to_array(img)\nimages.append(imread(img_path))\ninputs.append(img.copy())\nimg_path = './pics/car_cat2.jpg'\nimg = image.load_img(img_path, target_size=(300, 300))\nimg = image.img_to_array(img)\nimages.append(imread(img_path))\ninputs.append(img.copy())\ninputs = preprocess_input(np.array(inputs))\n"

In [ ]:
preds = model.predict(inputs, batch_size=1, verbose=1)

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step


In [ ]:
results = bbox_util.detection_out(preds)

In [ ]:
%%time
a = model.predict(inputs, batch_size=1)
b = bbox_util.detection_out(preds)

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
CPU times: user 786 ms, sys: 44.5 ms, total: 831 ms
Wall time: 756 ms


In [ ]:
for i, img in enumerate(images):
    # Parse the outputs.
    det_label = results[i][:, 0]
    det_conf = results[i][:, 1]
    det_xmin = results[i][:, 2]
    det_ymin = results[i][:, 3]
    det_xmax = results[i][:, 4]
    det_ymax = results[i][:, 5]

    # Get detections with confidence higher than 0.6.
    top_indices = [i for i, conf in enumerate(det_conf) if conf >= threshold]

    top_conf = det_conf[top_indices]
    top_label_indices = det_label[top_indices].tolist()
    top_xmin = det_xmin[top_indices]
    top_ymin = det_ymin[top_indices]
    top_xmax = det_xmax[top_indices]
    top_ymax = det_ymax[top_indices]

    colors = plt.cm.hsv(np.linspace(0, 1, 21)).tolist()

    plt.imshow(img / 255.)
    currentAxis = plt.gca()

    for i in range(top_conf.shape[0]):
        xmin = int(round(top_xmin[i] * img.shape[1]))
        ymin = int(round(top_ymin[i] * img.shape[0]))
        xmax = int(round(top_xmax[i] * img.shape[1]))
        ymax = int(round(top_ymax[i] * img.shape[0]))
        score = top_conf[i]
        label = int(top_label_indices[i])
        label_name = voc_classes[label - 1]
        display_txt = '{:0.2f}, {}'.format(score, label_name)
        coords = (xmin, ymin), xmax-xmin+1, ymax-ymin+1
        color = colors[label]
        currentAxis.add_patch(plt.Rectangle(*coords, fill=False, edgecolor=color, linewidth=2))
        currentAxis.text(xmin, ymin, display_txt, bbox={'facecolor':color, 'alpha':0.5})

    plt.show()

Output hidden; open in https://colab.research.google.com to view.